# Introduction

This is a notebook to be used solely for inference, applying a trained spacy model we train in src.ipynb to new documents inside respective folders

In [9]:
import spacy
import pickle
import pandas as pd
import os 

## Utilities functions

In [10]:
# Utilities
def create_pattern(text: str) -> list:
    try:
        tokens = text.split()
        # Even if there's only one token, we still return a list of dictionaries.
        return [{"LOWER": str.lower(token)} for token in tokens]
    except:
        pass

def get_entities_fr_text(text: str) -> list:
    try:
        result = trained_model(text)
        return [(ent.text, ent.label_) for ent in result.ents]
    except ValueError as e:
        if "[E088]" in str(e):  
            print("chunking text")
            text_chunks = [text[:99999], text[1000000:]] #hardcoding 2 chunks. EDIT to be more flexible.
            result_list = [trained_model(chunk) for chunk in text_chunks]
            final_result = []
            for result in result_list:
                final_result.extend([(ent.text, ent.label_) for ent in result.ents])
            return final_result
        else:
            print(f"Other ValueError: {e}")

def print_results_content(results_dict: dict) -> None:
    print(f"""
    # of fund_managers unique: {len(set(results_dict["fund_managers"]))}
    # of fund_managers duplicate: {len(results_dict["fund_managers"])}
    # of funds unique: {len(set(results_dict["investments"]))}
    # of funds duplicate: {len(results_dict["investments"])}
    """)

def get_aggregate_inference() -> dict:
    """
    Assumes a pipeline component has been added/removed.
    Gets inference results with that change included.
    """
    results_dict = {}
    for text, _ in TRAIN_DATA + TEST_DATA:
        ents_list = get_entities_fr_text(text)
        for ent in ents_list:
            ent, ent_category = ent[0], ent[1]
            if ent_category not in results_dict:
                results_dict[ent_category] = [ent]
            else:
                results_dict[ent_category].append(ent)
    return results_dict

# define a function to retrieve predictions (entity list) into a dictionary for accounting
entity_results = {}
def entity_list_into_entity_results(input_list: list) -> None:
    results_dict = {}
    for ent_tuple in input_list:
        ent_text, ent_label = ent_tuple[0], ent_tuple[1]
        if ent_label not in results_dict:
            results_dict[ent_label] = [ent_text]
        else:
            results_dict[ent_label].append(ent_text)
    return results_dict

In [11]:
# establish input data directories
input_directory = '../data/raw'
labels_directory = '../data/processed/pre_processing/llm_entities'
matching_folder = '../data/matching'

In [6]:
# initialise trained spacy NER model
trained_model = spacy.load("../models/model-best")

In [13]:
# load in the funds and fund managers from MDM

df_mdm = pd.read_excel(f"{matching_folder}/FundMaster.xlsx", sheet_name=0)

# load into sets so there are no duplicates
fund_managers, funds = set(), set()
funds.update(df_mdm['Fund Name'].unique().tolist())
fund_managers.update(df_mdm['Fund Manager'].unique().tolist())

In [14]:
# Add the matcher to the trained spaCy model
# Add the EntityRuler component before the 'ner' component. this component does case insensitive token matching

if "entity_ruler" in trained_model.pipe_names:
    trained_model.remove_pipe("entity_ruler")

entity_ruler = trained_model.add_pipe("entity_ruler", after="ner", config={"overwrite_ents": False})

patterns = [{"label": "investments", "pattern": create_pattern(fund)} for fund in funds] + \
[{"label": "fund_managers", "pattern": create_pattern(fund_mgr)} for fund_mgr in fund_managers]

# Add patterns to the entity ruler
entity_ruler.add_patterns(patterns)

print(f"nlp pipeline components: {trained_model.pipe_names}")

nlp pipeline components: ['tok2vec', 'ner', 'entity_ruler']


# Inference step

This step actually performs inference in the `get_entities_fr_text()` function

In [15]:
# loop through files in my input directory, retrieve entities, store retrieved entities in a dictionary
for filename in os.listdir(input_directory):
    if filename.endswith(".txt"):
        filepath = os.path.join(input_directory, filename)

        print(f"processing filename: {filename}")
        with open(filepath, "r", encoding="utf-8") as file:
            content = file.read()

            # inference performed here
            ent_list = get_entities_fr_text(content)

            #parse this into an entity dictionary
            if filename not in entity_results:
                entity_results[filename] = entity_list_into_entity_results(ent_list)

processing filename: 01 - MRC - Action Item register 2024 (A1224500).txt
processing filename: 01 ARC - Decision item register 2024 (A1136154).txt
processing filename: 01 IC - Action item register 2024 (A1108669).txt
processing filename: 01 IC - Annual calendar 2024 (A1081726).txt
processing filename: ARC & MRC - 20241122 - Minutes (A604869) (A1230142).txt
processing filename: ARC & MRC - 20241122 - Papers (A1086718) (A1229970).txt
processing filename: ARC - 20240507 - Minutes (FINAL) (A1164356).txt
processing filename: ARC - 20240507 - Papers (A1164349).txt
processing filename: ARC - 20240620 - Minutes (FINAL) (A1179933).txt
processing filename: ARC - 20240620 - Papers (A1179932).txt
processing filename: ARC - 20240815 - Minutes (FINAL) (A1203706).txt
processing filename: ARC - 20240815 - Papers (A1203771).txt
processing filename: ARC - 20240903 - Minutes (Draft) (A604869) (A1209556).txt
processing filename: ARC - 20240903 - Papers (A1208832).txt
processing filename: IC - 20240919- Min

## Accounting

Count fund and fund manager entities across varied sets of documents to gauge performance

In [21]:
funds, fund_managers = set(), set()
for file, entities_dict in entity_results.items():
    if 'fund_managers' in entities_dict:
        fund_managers.update([manager for manager in entities_dict['fund_managers']])
    elif 'investments' in entities_dict:
        funds.update([fund for fund in entities_dict['investments']])

In [22]:
# compare funds and fund managers that are in those actual documents - THIS IS FOR FILES WITH LABELS
# loop through files in my directory, store retrieved entities in a dictionary
fund_label_set, fund_manager_label_set = set(), set()
raw_label_dict = {}

for filename in os.listdir(labels_directory):
    if filename.endswith(".csv"):
        filepath = os.path.join(labels_directory, filename)

        print(f"processing filename: {filename}")
        if filename not in raw_label_dict:
            df_labels = pd.read_csv(filepath, delimiter=",", quotechar='"', on_bad_lines='skip')
            raw_label_dict[filename] = df_labels.to_json()

            fund_manager_label_set.update(list(df_labels['fund_manager'].values))
            fund_label_set.update(list(df_labels['investments'].values))

processing filename: 01 ARC - Decision item register 2024 (A1136154) copy.csv
processing filename: 01 IC - Action item register 2024 (A1108669) copy.csv
processing filename: ARC & MRC - 20241122 - Minutes (A604869) (A1230142) copy.csv
processing filename: ARC & MRC - 20241122 - Papers (A1086718) (A1229970) copy.csv
processing filename: ARC - 20240507 - Minutes (FINAL) (A1164356) copy.csv
processing filename: ARC - 20240620 - Papers (A1179932) copy.csv
processing filename: ARC - 20240815 - Minutes (FINAL) (A1203706) copy.csv
processing filename: IC - 20241023 - Minutes (FINAL) (A1222441) copy.csv
processing filename: IC - 20241023 - Papers (A1222038) copy.csv


In [23]:
# print out funds, fund managers captured by spacy, number of input docs
num_input_docs = len(entity_results.keys())

print(f"number of funds extracted by spacy in {num_input_docs} docs: {len(funds)}")
print(f"number of fund managers extracted by spacy in {num_input_docs} docs: {len(fund_managers)}")

number of funds extracted by spacy in 9 docs: 4
number of fund managers extracted by spacy in 9 docs: 18


In [24]:
# print out examples of funds and fund managers
print(f"fund examples: {funds}")
print(f"fund manager examples: {fund_managers}")

fund examples: {'Jemena', 'Port. Strat', 'State Grid', 'Zinfra'}
fund manager examples: {'ATLAS Infrastructure', 'CDC', 'FF Internal', 'APAC', 'Silver Point Capital', 'Lightspeed Venture Partners', 'Blackrock Alternative Advisors', 'internal', 'Effissimo Capital Management', 'Greystar', 'M & G', 'Omers', 'Blackrock', 'Port of Melbourne', 'Internal', 'Man Group', 'OMERS', 'EQT Infrastructure'}


In [25]:
# print out recall metrics where relevant in the test set
print(f"% recall of funds by spacy in {num_input_docs} docs: {len(fund_label_set.intersection(funds))/len(funds)}")
print(f"% recall of fund managers by spacy in {num_input_docs} docs: {len(fund_manager_label_set.intersection(fund_managers))/len(fund_managers)}")

% recall of funds by spacy in 9 docs: 0.25
% recall of fund managers by spacy in 9 docs: 0.2777777777777778
